# PROJECT STAGE

Current stage: Research exploration

Code may later be migrated to src modules once stabilized.

# 1. Notebook Metadata

In [ ]:
# ============================================================
# NOTEBOOK METADATA
# ============================================================

NOTEBOOK_NAME = "01_data_pipeline"
AUTHOR = "Juan Manuel Giacame"
CREATED = "2026-03-15"
UPDATED = "2026-03-15"
PROJECT = "macro-market-analysis"
STAGE = "Data Processing"
VERSION = "0.1.0"
GIT_HASH = ""  # completar con commit hash si se usa git
EXPERIMENT_ID = f"{NOTEBOOK_NAME}_{CREATED}_{VERSION}"

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
PURPOSE = """
Transform raw market data into a clean processed dataset suitable
for research and feature engineering.

The pipeline performs:

    - Raw data validation
    - Price adjustment for dividends, capital gains and stock splits
    - Construction of Total Return prices
    - Log return calculation
    - Liquidity feature computation
    - Cross-asset time alignment
    - Missing data handling

Processed datasets are stored per asset in:

data/processed/{asset}.parquet
"""

# ------------------------------------------------------------
# INPUT DATASETS
# ------------------------------------------------------------
INPUT_DATASETS = [
    "data/raw/{asset}.parquet",
    "Universe defined in src/quant_research/data/registry/universe_registry.py"
]

# ------------------------------------------------------------
# OUTPUT DATASETS
# ------------------------------------------------------------
OUTPUT_DATASETS = [
    "data/processed/{asset}.parquet"
]

# ------------------------------------------------------------
# DATASET DESCRIPTION
# ------------------------------------------------------------
DATASET_DESCRIPTION = """
The processed dataset represents the canonical market dataset
used by the research layer.

Each asset dataset contains:

    - Raw OHLCV market data
    - Corporate actions (Dividends, Stock Splits, Capital Gains)
    - Adjusted close price (adj_close)
    - Log returns (log_ret)
    - Liquidity proxy (dollar_volume)

Adjusted prices incorporate dividends, capital gains and stock splits
and reproduce the total return price series used for return calculations.
"""

# ------------------------------------------------------------
# ASSETS UNIVERSE
# ------------------------------------------------------------
ASSETS_UNIVERSE = "All assets defined in universe_registry.py (configurable)"

# ------------------------------------------------------------
# DEPENDENCIES
# ------------------------------------------------------------
DEPENDENCIES = ["pandas >= 2.0", "numpy", "pathlib", "pyarrow"]

# ------------------------------------------------------------
# NOTES
# ------------------------------------------------------------
NOTES = """
This notebook represents the core market data processing layer.

Key design principles:

    - Raw data remains immutable
    - All adjustments are applied in the processing layer
    - Log returns are used as the canonical return representation
    - Adjusted close prices are used for momentum features
    - Processed datasets serve as the base input for feature pipelines

This notebook intentionally keeps the processing logic explicit
for research transparency. Later stages may refactor the logic
into reusable pipeline modules inside src/quant_research.
"""


# 2. Imports

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

# Project configuration
from quant_research.config.paths import RAW_DATA_PATH, PROCESSED_DATA_PATH

# Asset universe
from quant_research.data.registry.universe_registry import ASSET_UNIVERSE

print(RAW_DATA_PATH)
print(PROCESSED_DATA_PATH)


In [ ]:
# ============================================================
# ASSET UNIVERSE
# ============================================================

print("Assets in universe:")

for asset in ASSET_UNIVERSE:
    print(asset.symbol)


In [ ]:
# ============================================================
# RAW DATA FILE CHECK
# ============================================================

for asset in ASSET_UNIVERSE:

    symbol = asset.symbol
    path = RAW_DATA_PATH / f"{symbol}.parquet"

    if not path.exists():
        print(f"Missing raw dataset: {symbol}")


# 3. Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# EXPECTED RAW DATA COLUMNS
# ------------------------------------------------------------

EXPECTED_COLUMNS = [
    "Open",
    "High",
    "Low",
    "Close",
    "Adj Close",
    "Volume",
    "Dividends",
    "Stock Splits",
    "Capital Gains"
]


# ------------------------------------------------------------
# CORPORATE ACTION COLUMNS
# ------------------------------------------------------------

CORPORATE_ACTION_COLUMNS = [
    "Dividends",
    "Stock Splits",
    "Capital Gains"
]


# ------------------------------------------------------------
# PRICE COLUMNS
# ------------------------------------------------------------

PRICE_COLUMNS = [
    "Open",
    "High",
    "Low",
    "Close"
]


# ------------------------------------------------------------
# PROCESSED DATASET COLUMNS
# ------------------------------------------------------------

PROCESSED_COLUMNS = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Dividends",
    "Stock Splits",
    "Capital Gains",
    "adj_close",
    "log_ret",
    "dollar_volume"
]


# ------------------------------------------------------------
# DATA CLEANING PARAMETERS
# ------------------------------------------------------------

# forward fill prices after alignment
PRICE_FILL_METHOD = "ffill"

# do not fill returns
RETURNS_FILL = False


# ------------------------------------------------------------
# RETURN CONFIGURATION
# ------------------------------------------------------------

RETURN_TYPE = "log"


# ------------------------------------------------------------
# CORPORATE ACTION ADJUSTMENT
# ------------------------------------------------------------

# combine distributions
INCLUDE_CAPITAL_GAINS = True


# ------------------------------------------------------------
# CROSS-ASSET ALIGNMENT
# ------------------------------------------------------------

ALIGN_ASSETS = True


# 4. Load Raw Market Data

In [ ]:
# ============================================================
# LOAD RAW MARKET DATA
# ============================================================

raw_data = {}

for asset in ASSET_UNIVERSE:

    symbol = asset.symbol
    path = RAW_DATA_PATH / f"{symbol}.parquet"

    print(f"\nLoading {symbol} from {path}")

    # --------------------------------------------------------
    # LOAD DATA
    # --------------------------------------------------------

    df = pd.read_parquet(path)

    # --------------------------------------------------------
    # RENAME VENDOR ADJUSTED PRICE
    # --------------------------------------------------------

    if "Adj Close" in df.columns:
        df = df.rename(columns={"Adj Close": "vendor_adj_close"})

    # --------------------------------------------------------
    # ENSURE DATETIME INDEX
    # --------------------------------------------------------

    df.index = pd.to_datetime(df.index)

    # --------------------------------------------------------
    # SORT CHRONOLOGICALLY
    # --------------------------------------------------------

    df = df.sort_index()

    # --------------------------------------------------------
    # CHECK DUPLICATED TIMESTAMPS
    # --------------------------------------------------------

    if df.index.duplicated().any():

        n_dup = df.index.duplicated().sum()

        print(f"⚠ {symbol} has {n_dup} duplicated timestamps")

        display(df[df.index.duplicated()].head())

        # remove duplicates (keep first)
        df = df[~df.index.duplicated(keep="first")]

        print(f"{symbol} duplicates removed")

    # --------------------------------------------------------
    # STORE DATASET
    # --------------------------------------------------------

    raw_data[symbol] = df


print("\nAssets loaded:")
print(list(raw_data.keys()))


In [ ]:
# ============================================================
# RAW DATA PREVIEW
# ============================================================

for symbol, df in raw_data.items():

    print("\n==============================")
    print(symbol)
    print("==============================")

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Start:", df.index.min())
    print("End:", df.index.max())

    display(df.head())


In [ ]:
# ============================================================
# VALIDATE RAW DATA COLUMNS
# ============================================================

for symbol, df in raw_data.items():

    missing_cols = [col for col in EXPECTED_COLUMNS if col not in df.columns]

    if len(missing_cols) > 0:
        print(f"{symbol} missing columns:", missing_cols)



In [ ]:
# ============================================================
# NORMALIZE CORPORATE ACTION COLUMNS
# ============================================================

for symbol, df in raw_data.items():

    for col in CORPORATE_ACTION_COLUMNS:

        if col not in df.columns:
            df[col] = 0.0

    raw_data[symbol] = df


# 5. Raw Data Diagnostics

In [ ]:
# ============================================================
# RAW DATA DIAGNOSTICS
# ============================================================

for symbol, df in raw_data.items():

    print("\n=================================================")
    print(f"DIAGNOSTICS: {symbol}")
    print("=================================================")

    # --------------------------------------------------------
    # BASIC INFO
    # --------------------------------------------------------

    print("\nDate range:")
    print(df.index.min(), "→", df.index.max())

    print("\nNumber of rows:", len(df))

    # --------------------------------------------------------
    # DUPLICATED INDEX CHECK
    # --------------------------------------------------------

    duplicated = df.index.duplicated().sum()
    print("\nDuplicated timestamps:", duplicated)

    # --------------------------------------------------------
    # DATE GAPS
    # --------------------------------------------------------

    date_diffs = df.index.to_series().diff().dropna()
    large_gaps = date_diffs[date_diffs > pd.Timedelta(days=3)]

    print("\nLarge gaps (>3 days):", len(large_gaps))

    if len(large_gaps) > 0:
        display(large_gaps.head())

    # --------------------------------------------------------
    # MISSING VALUES
    # --------------------------------------------------------

    print("\nMissing values:")
    print(df.isna().sum())

    # --------------------------------------------------------
    # DIVIDENDS
    # --------------------------------------------------------

    if "Dividends" in df.columns:

        div_events = df[df["Dividends"] > 0]

        print("\nDividend events:", len(div_events))

        if len(div_events) > 0:
            display(div_events[["Close", "Dividends"]].head())

    # --------------------------------------------------------
    # STOCK SPLITS
    # --------------------------------------------------------

    if "Stock Splits" in df.columns:

        split_events = df[df["Stock Splits"] != 0]

        print("\nStock split events:", len(split_events))

        if len(split_events) > 0:
            display(split_events[["Close", "Stock Splits"]])

    # --------------------------------------------------------
    # CAPITAL GAINS
    # --------------------------------------------------------

    if "Capital Gains" in df.columns:

        cap_gain_events = df[df["Capital Gains"] > 0]

        print("\nCapital gains events:", len(cap_gain_events))

        if len(cap_gain_events) > 0:
            display(cap_gain_events[["Close", "Capital Gains"]])

    # --------------------------------------------------------
    # EXTREME DIVIDENDS
    # --------------------------------------------------------

    if "Dividends" in df.columns:

        extreme_div = df[df["Dividends"] > df["Close"] * 0.2]


        print("\nExtrem Dividend events:", len(extreme_div))

        if len(extreme_div) > 0:
            display(div_events[["Close", "Dividends"]].head())

    


# 6. CORE COMPUTATIONS

## 6.1 Corporate Actions Adjustment

This section constructs a **Total Return Adjusted Price** series from the raw market data.  
The objective is to produce a price series suitable for **return computation and feature engineering** while maintaining **point-in-time correctness**.

The adjustment incorporates **distributions** such as dividends and capital gains.

Stock splits are **not re-applied**, since the OHLC prices provided by Yahoo Finance (via yfinance) are already **split-adjusted**.

---

### Distribution Construction

Distributions are defined as the sum of dividends and capital gains:

$$
\[
D_t = Dividends_t + CapitalGains_t
\]
$$

If capital gains are not present:

$$
\[
D_t = Dividends_t
\]
$$

---

### Distribution Adjustment Factor

To incorporate distributions into historical prices, a **distribution factor** is constructed.

For each date $\( t \)$:

$$
\[
f_t =
\begin{cases}
\frac{P_{t-1} - D_t}{P_{t-1}}, & \text{if } D_t > 0 \\
1, & \text{otherwise}
\end{cases}
\]
$$

where:

- $\( P_{t-1} \)$ is the previous day's closing price  
- $\( D_t \)$ is the distribution occurring at time $\( t \)$

This formulation ensures that the adjustment uses **only past information**, preventing look-ahead bias.

---

### Cumulative Adjustment Factor

Adjustment factors are accumulated **backwards through time** to correctly adjust historical prices.
$$
\[
A_t = \prod_{i=t+1}^{T} f_i
\]
$$
where:

- $\( T \)$ is the most recent observation
- factors are applied **from the present to the past**

This produces a **point-in-time consistent adjustment factor**.

---

### Adjusted Price Construction

The adjusted close price is obtained by applying the cumulative adjustment factor to the raw close price.
$$
\[
P_t^{adj} = P_t \cdot A_t
\]
$$
where:

-$ \( P_t \)$ is the raw close price  
- $\( A_t \)$ is the cumulative adjustment factor

The resulting price series represents a **Total Return Price**.

---

### Properties

The adjusted price series has the following properties:

1. **Continuity across distributions**  
   Dividend payments do not produce artificial price drops.

2. **Point-in-time consistency**  
   Adjustments depend only on information available at time \( t \).

3. **Compatibility with return computation**  
   Log returns derived from the adjusted price represent total returns:
$$
\[
r_t = \ln\left(\frac{P_t^{adj}}{P_{t-1}^{adj}}\right)
\]
$$
---

### Output Columns Generated

This section adds the following columns to each asset dataset:

| Column | Description |
|------|------|
| distribution | Dividends + capital gains |
| dist_factor | Distribution adjustment factor |
| cum_adj_factor | Cumulative adjustment factor |
| adj_close | Total return adjusted price |

These adjusted prices will be used as the **base price series for return computation and feature engineering** in later sections of the pipeline.

In [ ]:
processed_data = {}

for symbol, df in raw_data.items():

    
    print(f"Processing corporate actions: {symbol}")
   

    df = df.copy()

    # --------------------------------------------------------
    # DISTRIBUTIONS (DIVIDENDS + CAPITAL GAINS)
    # --------------------------------------------------------

    df["distribution"] = 0.0

    if "Dividends" in df.columns:
        df["distribution"] += df["Dividends"].fillna(0)

    if "Capital Gains" in df.columns:
        df["distribution"] += df["Capital Gains"].fillna(0)

    # --------------------------------------------------------
    # DISTRIBUTION FACTOR
    # --------------------------------------------------------

    df["dist_factor"] = 1.0

    mask = df["distribution"] > 0

    df.loc[mask, "dist_factor"] = (
        (df["Close"].shift(1) - df["distribution"]) /
        df["Close"].shift(1)
    )

    df["dist_factor"] = df["dist_factor"].fillna(1)

    # --------------------------------------------------------
    # CUMULATIVE FACTOR
    # --------------------------------------------------------

    df["cum_adj_factor"] = df["dist_factor"][::-1].cumprod()[::-1]

    # --------------------------------------------------------
    # TOTAL RETURN ADJUSTED PRICE
    # --------------------------------------------------------

    df["adj_close"] = df["Close"] * df["cum_adj_factor"]

    processed_data[symbol] = df

print("\nProcessed assets:")
print(list(processed_data.keys()))

In [ ]:
print("Processed assets:")
print(list(processed_data.keys()))


In [ ]:
df = processed_data["IWM"]

df[["Close","vendor_adj_close","adj_close"]].plot(figsize=(12,6))


In [ ]:
print("Raw assets:", list(raw_data.keys()))
print("Processed assets:", list(processed_data.keys()))

In [ ]:
# ============================================================
# CORPORATE ACTION SANITY CHECK
# ============================================================

import pandas as pd
import numpy as np

spike_threshold = 0.30   # 30% move

issues = []

for symbol, df in processed_data.items():

    returns = df["adj_close"].pct_change()

    spikes = returns.abs() > spike_threshold

    if spikes.sum() > 0:

        spike_dates = df.index[spikes]

        issues.append({
            "asset": symbol,
            "spike_count": spikes.sum(),
            "max_move": returns.abs().max(),
            "dates": spike_dates[:5]
        })

if issues:
    print("Potential corporate action issues detected:\n")
    display(pd.DataFrame(issues))
else:
    print("No abnormal return spikes detected.")

In [ ]:
# ============================================================
# VENDOR CONSISTENCY TEST
# ============================================================

tolerance = 0.001

mismatches = []

for symbol, df in processed_data.items():

    if "vendor_adj_close" not in df.columns:
        continue

    ratio = df["adj_close"] / df["vendor_adj_close"]

    deviation = (ratio - 1).abs()

    if deviation.max() > tolerance:

        mismatches.append({
            "asset": symbol,
            "max_deviation": deviation.max()
        })

if mismatches:
    print("Vendor adjustment mismatch detected:\n")
    display(pd.DataFrame(mismatches))
else:
    print("Adjustment matches vendor data.")

In [ ]:
# ============================================================
# SPLIT CONSISTENCY TEST
# ============================================================

split_issues = []

for symbol, df in raw_data.items():

    if "Stock Splits" not in df.columns:
        continue

    splits = df[df["Stock Splits"] > 0]

    for date, row in splits.iterrows():

        try:

            prev_close = df.loc[:date]["Close"].iloc[-2]
            curr_close = df.loc[:date]["Close"].iloc[-1]

            observed_ratio = prev_close / curr_close
            expected_ratio = row["Stock Splits"]

            if abs(observed_ratio - expected_ratio) > 0.1:

                split_issues.append({
                    "asset": symbol,
                    "date": date,
                    "expected_split": expected_ratio,
                    "observed_ratio": observed_ratio
                })

        except:
            pass

if split_issues:
    print("Split inconsistencies detected:\n")
    display(pd.DataFrame(split_issues))
else:
    print("All splits consistent with price changes.")

In [ ]:
import numpy as np

tracking = []

for symbol, df in processed_data.items():

    if "vendor_adj_close" not in df.columns:
        continue

    r1 = np.log(df["adj_close"]).diff()
    r2 = np.log(df["vendor_adj_close"]).diff()

    diff = (r1 - r2).dropna()

    tracking.append({
        "asset": symbol,
        "tracking_error": diff.std()
    })

pd.DataFrame(tracking).sort_values("tracking_error", ascending=False)

In [ ]:
for symbol, df in processed_data.items():

    if (df["adj_close"] <= 0).any():
        print(f"Negative price detected: {symbol}")

## 6.2 Log Returns

This section computes **logarithmic returns** from the adjusted price series.

Returns are calculated using the **total return adjusted price (`adj_close`)** constructed in the previous step.  
This ensures that distributions such as dividends and capital gains are properly incorporated into the return series.

---

### Definition

The daily log return is defined as:
$$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$
where:

- $ P_t $ is the adjusted price at time $ t $
- $ P_{t-1} $ is the adjusted price at time $ t-1 $

Logarithmic returns are preferred in quantitative finance because they have useful statistical properties.

---

### Properties of Log Returns

Log returns have several advantages over simple returns:

1. **Time Additivity**

Returns over longer horizons can be obtained by summing shorter horizon returns:
$$
r_{t,t+n} = \sum_{i=1}^{n} r_{t+i}
$$
2. **Statistical Stability**

Log returns tend to exhibit more stable statistical behavior, making them suitable for:

- volatility estimation
- risk models
- regime detection

3. **Symmetry**

Log returns treat upward and downward movements more symmetrically compared to simple returns.

---

### Multi-Horizon Returns

In addition to the daily return, the pipeline computes **log returns across multiple horizons**.

For a horizon $( h ):$
$$
r_{t,h} = \ln\left(\frac{P_t}{P_{t-h}}\right)
$$
These horizons correspond approximately to common financial timeframes:

| Horizon | Approximate Period |
|------|------|
| 5 | 1 week |
| 21 | 1 month |
| 63 | 3 months |
| 126 | 6 months |
| 252 | 1 year |

These return horizons are useful for:

- risk estimation
- volatility modeling
- exploratory research

Note that **momentum features will be constructed separately**, as momentum signals typically exclude the most recent month to avoid short-term reversal effects.

---

### Output Columns Generated

This section adds the following columns to each dataset:

| Column | Description |
|------|------|
| log_ret | Daily log return |
| log_ret_5 | 1-week log return |
| log_ret_21 | 1-month log return |
| log_ret_63 | 3-month log return |
| log_ret_126 | 6-month log return |
| log_ret_252 | 12-month log return |

These return series will serve as **core inputs for volatility features, regime models, and exploratory research**.

In [ ]:
# ============================================================
# RETURN COMPUTATION
# ============================================================

print("\nComputing returns...\n")

for symbol, df in processed_data.items():

    print(f"Processing returns: {symbol}")

    df = df.copy()

    # --------------------------------------------------------
    # DAILY LOG RETURNS
    # --------------------------------------------------------

    df["log_ret"] = np.log(df["adj_close"] / df["adj_close"].shift(1))

    # --------------------------------------------------------
    # MULTI-HORIZON LOG RETURNS
    # --------------------------------------------------------

    horizons = [5, 21, 63, 126, 252]

    for h in horizons:

        df[f"log_ret_{h}"] = np.log(df["adj_close"] / df["adj_close"].shift(h))

    # --------------------------------------------------------
    # STORE RESULT
    # --------------------------------------------------------

    processed_data[symbol] = df


print("\nReturn computation completed.")

In [ ]:
df = processed_data["SPY"]

df[["log_ret"]].describe()

## 6.3 Liquidity Features

This section computes simple **liquidity proxies** from market data.

Liquidity is an important dimension in quantitative research because it affects:

- transaction costs
- strategy capacity
- signal reliability

Highly illiquid assets can produce **unstable signals** and unrealistic backtest results.

---

### Dollar Volume

The primary liquidity proxy used in this pipeline is **dollar volume**, defined as:
$$
DV_t = P_t \times V_t
$$


where:

- $(P_t)$ is the closing price
- $(V_t)$ is the traded volume

Dollar volume represents the **notional value traded in a given period**.

Higher dollar volume generally implies:

- deeper markets
- lower market impact
- better trade execution conditions

---

### Rolling Liquidity Measures

To smooth daily fluctuations, rolling liquidity metrics are often used.

For a rolling window of length $(n)$:

$$
DV_{t,n} = \frac{1}{n} \sum_{i=0}^{n-1} DV_{t-i}
$$

Typical windows include:

| Window | Interpretation |
|------|------|
| 21 | Monthly liquidity |
| 63 | Quarterly liquidity |

These rolling measures help identify persistent changes in liquidity conditions.

---

### Role in the Research Pipeline

Liquidity metrics can be used to:

- filter illiquid assets
- normalize signals
- construct liquidity regimes
- improve backtest realism

Even in a small universe, monitoring liquidity provides useful diagnostic information about market structure.

---

### Output Columns Generated

This section adds the following liquidity-related columns:

| Column | Description |
|------|------|
| dollar_volume | Daily traded notional value |
| dollar_volume_21 | 1-month average dollar volume |
| dollar_volume_63 | 3-month average dollar volume |

These features may later be used to **filter assets or analyze liquidity regimes in the research layer**.

In [ ]:
# ============================================================
# LIQUIDITY FEATURES
# ============================================================

print("\nComputing liquidity features...\n")

for symbol, df in processed_data.items():

    print(f"Processing liquidity: {symbol}")

    df = df.copy()

    # --------------------------------------------------------
    # DOLLAR VOLUME
    # --------------------------------------------------------

    df["dollar_volume"] = df["Close"] * df["Volume"]

    # --------------------------------------------------------
    # ROLLING LIQUIDITY
    # --------------------------------------------------------

    df["dollar_volume_21"] = df["dollar_volume"].rolling(21).mean()
    df["dollar_volume_63"] = df["dollar_volume"].rolling(63).mean()

    # --------------------------------------------------------
    # STORE RESULT
    # --------------------------------------------------------
    # remove column index name inherited from vendor
    df.columns.name = None
    processed_data[symbol] = df

print("\nLiquidity feature computation completed.")

In [ ]:
df = processed_data["SPY"]

df[["dollar_volume"]].describe()

In [ ]:
df = processed_data["SPY"]

print(df.columns)

In [ ]:
for symbol, df in processed_data.items():

    print(symbol)

    print("duplicates:", df.index.duplicated().sum())
    print("missing adj_close:", df["adj_close"].isna().sum())
    print("max return:", df["log_ret"].abs().max())

    print()

In [ ]:
column_order = [
"Open","High","Low","Close","Volume",
"vendor_adj_close",
"Dividends","Capital Gains","Stock Splits",
"distribution","dist_factor","cum_adj_factor","adj_close",
"log_ret","log_ret_5","log_ret_21","log_ret_63","log_ret_126","log_ret_252",
"dollar_volume","dollar_volume_21","dollar_volume_63"
]

df = df[column_order]

# 7. Export

In [ ]:
# ============================================================
# SAVE PROCESSED DATASETS (INCREMENTAL)
# ============================================================

print("\nSaving processed datasets (incremental)...\n")

for symbol, df_new in processed_data.items():

    output_path = PROCESSED_DATA_PATH / f"{symbol}.parquet"

    df_new = df_new.sort_index()

    # --------------------------------------------------------
    # IF DATASET EXISTS → MERGE
    # --------------------------------------------------------

    if output_path.exists():

        df_existing = pd.read_parquet(output_path)

        df_combined = pd.concat([df_existing, df_new])

        # remove duplicate dates
        df_combined = df_combined[~df_combined.index.duplicated(keep="last")]

        df_combined = df_combined.sort_index()

        df_combined.to_parquet(output_path)

        print(f"Updated: {symbol} → {output_path}")

    # --------------------------------------------------------
    # FIRST TIME SAVE
    # --------------------------------------------------------

    else:

        df_new.to_parquet(output_path)

        print(f"Created: {symbol} → {output_path}")

print("\nProcessed datasets saved successfully.")

# 8. Data Audit & Validation

In [ ]:
# ============================================================
# DATA AUDIT & VALIDATION
# ============================================================

print("\nRunning processed data audit...\n")

audit_rows = []

for file in PROCESSED_DATA_PATH.glob("*.parquet"):

    symbol = file.stem

    df = pd.read_parquet(file)

    start_date = df.index.min()
    end_date = df.index.max()

    rows = len(df)

    duplicates = df.index.duplicated().sum()

    missing_prices = df["Close"].isna().sum()
    missing_adj = df["adj_close"].isna().sum()

    max_return = df["log_ret"].abs().max()

    avg_dollar_volume = df["dollar_volume"].mean()

    audit_rows.append({
        "asset": symbol,
        "start_date": start_date,
        "end_date": end_date,
        "rows": rows,
        "duplicate_dates": duplicates,
        "missing_close": missing_prices,
        "missing_adj_close": missing_adj,
        "max_abs_return": max_return,
        "avg_dollar_volume": avg_dollar_volume
    })

audit_df = pd.DataFrame(audit_rows)

audit_df = audit_df.sort_values("asset")

audit_df

In [ ]:
required_columns = [
"Open","High","Low","Close","Volume",
"vendor_adj_close",
"Dividends","Capital Gains","Stock Splits",
"distribution","dist_factor","cum_adj_factor","adj_close",
"log_ret","log_ret_5","log_ret_21","log_ret_63","log_ret_126","log_ret_252",
"dollar_volume","dollar_volume_21","dollar_volume_63"
]

for file in PROCESSED_DATA_PATH.glob("*.parquet"):

    df = pd.read_parquet(file)

    missing = set(required_columns) - set(df.columns)

    if missing:
        print(file.stem, "missing:", missing)

In [ ]:
symbol = "SPY"

df = pd.read_parquet(PROCESSED_DATA_PATH / f"{symbol}.parquet")

df[["Close","adj_close"]].plot(figsize=(12,6))

In [ ]:


symbol = "SPY"

df = pd.read_parquet(PROCESSED_DATA_PATH / f"{symbol}.parquet")

df["log_ret"].abs().plot(
    figsize=(12,4),
    title=f"{symbol} Absolute Daily Returns"
)

In [ ]:
max_returns = {}

for file in PROCESSED_DATA_PATH.glob("*.parquet"):

    symbol = file.stem
    df = pd.read_parquet(file)

    max_returns[symbol] = df["log_ret"].abs().max()

pd.Series(max_returns).sort_values(ascending=False)

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

processed_path = Path("data/processed")

assets = [f.stem for f in PROCESSED_DATA_PATH.glob("*.parquet")]

for asset in assets:
    df = pd.read_parquet(PROCESSED_DATA_PATH / f"{asset}.parquet")
    
    # Reset index
    df_reset = df.reset_index()
    date_col = df_reset.columns[0]

    # Crear un dataframe largo solo con los gráficos que corresponden
    # Cada fila tendrá su propio eje y
    df_long = pd.melt(
        df_reset,
        id_vars=date_col,
        value_vars=['Close', 'adj_close', 'log_ret', 'dollar_volume'],
        var_name='feature',
        value_name='value'
    )

    # Asignamos la fila del subplot
    df_long['facet_row'] = df_long['feature'].map({
        'Close': 1,
        'adj_close': 1,
        'log_ret': 2,
        'dollar_volume': 3
    })

    # Colores consistentes
    color_map = {
        'Close': 'blue',
        'adj_close': 'red',
        'log_ret': 'orange',
        'dollar_volume': 'green'
    }

    fig = px.line(
        df_long,
        x=date_col,
        y='value',
        color='feature',
        facet_row='facet_row',
        color_discrete_map=color_map,
        title=f"{asset} - Interactive Visualization",
        height=900
    )

    # Sincronizar zoom en x
    fig.update_xaxes(matches='x')
    # Escala independiente en y
    fig.update_yaxes(matches=None)
    
    # Layout limpio
    fig.update_layout(showlegend=True)
    fig.show()